# EMG Data Pre-processing Script 

This script loops through the raw EMG files, preprocesses the raw data, and extracts variance, WL, and RMS using a sliding window. The preprocessed data can be saved to an Excel file, and the results can be visualized with a GUI.

## Library Importations 

In [ ]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd
%matplotlib qt
import matplotlib
matplotlib.use('QtAgg') 
from matplotlib import pyplot as plt


## Defining initial variables 

In [ ]:
to_keep=['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2','EOG1','EOG2','Corr', 'Zygo', 'Menton','Trigger']
eeg_ch= ['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2']
emg_ch= ['Corr', 'Zygo', 'Menton'] # Menton is chin EMG for sleep scoring
eog_ch= ['EOG1', 'EOG2']
trigger_ch= ['Trigger']

inter_trigger_length = 30
num_epochs = 60 # 60 epochs for each session 

raw_path= "/Users/zeynepozkaya/Desktop/SoundSleep/Python_Scripts/Data/SoundSleep_participants/"
current_index=0
inter_trigger_length=10
window = 50 
step = 1 

## Helper functions 

In [ ]:
# pre-processes data for each subject and returns df with session divided into 60 epochs w meta-data attached 
def pre_process_subjets(subject,filename):
    global raw_path
    global frq

    raw = mne.io.read_raw_brainvision(raw_path+subject+"/"+filename, preload=True)
    emg_ch= ['Zygo', 'Menton']


    if subject=='Cami':  # For Cami, EOG was recorded on IO channel
        mne.rename_channels(info=raw.info,mapping={'66':'ECG' ,'67':'Menton','68':'Zygo'})
        raw.set_channel_types(mapping={'ECG':'ecg','Zygo':'emg','Menton':'emg','IO':'eog'})
        mne.add_reference_channels(raw, ref_channels=['C3'], copy=False)
    else:
        mne.rename_channels(info=raw.info,mapping={'66':'ECG' ,'67':'Menton','68':'Zygo','69':'EOG'})
        raw.set_channel_types(mapping={'ECG':'ecg','Zygo':'emg','Menton':'emg','IO':'eog','EOG':'eog'})  
        mne.add_reference_channels(raw, ref_channels=['Cz'], copy=False)  
    if subject=='S02':
        idx=raw.ch_names.index('EOG')
        raw._data[idx]*=-1
        
    #raw = mne.set_bipolar_reference(raw, anode='IO', cathode='AF7',drop_refs=False) #
    raw.resample(sfreq=250)

    #raw.set_eeg_reference(['TP10'], projection=False) 

    emg_filter_params = {'lpass': 100,'hpass': 10,'notches': [50]}

    raw.filter(l_freq=emg_filter_params['hpass'],h_freq=emg_filter_params['lpass'],picks=emg_ch)
    raw.notch_filter(emg_filter_params['notches'],picks=emg_ch)

    events_all= mne.events_from_annotations(raw)[0]
    events= mne.pick_events(events_all,include=list(range(1,40))+[99]) # Pick events of interest (where a stimulus was presented)

    df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
    df_triggers.drop(columns=['dunno'],inplace=True)
    df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


    # Create trial type column 
    conditions = [
        df_triggers['Trigger'].between(1,10),
        df_triggers['Trigger'].between(11,20),
        df_triggers['Trigger'].between(21,30),
        df_triggers['Trigger'].between(31,40),
        df_triggers['Trigger'] == 99 
    ]

    choices = ['SNR1','SNR2','SNR3','blank', 'Tone'] # trial types 
    df_triggers['trial_type'] = np.select(conditions, choices, default='unknown')

    epochs = mne.Epochs(raw, events, tmin=-0, tmax=7,baseline=None, detrend=0,
                    reject=None, preload=True, on_missing='warn')

    
    return epochs, df_triggers 

In [ ]:
# extracts features in a sliding window of size window samples with a step size of a certain number of samples 
def get_features(epoch):
     # pad epoch to preserve sample number 
     pad_left  = window // 2
     pad_right = window - 1 - pad_left   
     
     epoch_padded = epoch
     epoch_padded = np.pad(epoch, (pad_left, pad_right), mode="edge")  

     # take sliding window 
     epoch_sw = np.lib.stride_tricks.sliding_window_view(epoch_padded,window)[::step]

     var = np.var(epoch_sw, axis=-1) # calculate variance over window 
     rms =  np.sqrt((1/window)*np.sum(epoch_sw**2, axis=-1)) # calculate rms over window  
     wl = np.sum(np.abs(np.diff(epoch_sw, axis=1)), axis=1) # calculate wl over window  

  
     return var, rms, wl 


In [ ]:
# function for looping through all files, pre-processing, performing feature extraction, and creating data frame to be used for classification 
def make_features_df(subject_epoch,subject):
    features = pd.DataFrame(
        index=range(num_epochs),
        columns=[
            "Subject",
            "Epochs", # epochs 
            # "Num_Contractions_Zygo", can add back in if have num contractions 
            "WL_Zygo", # three features being used 
            "Var_Zygo",
            "RMS_Zygo", 
            "Zygo", # processed EMG signal for epoch 
        ]
        )   
    
    for t in range(len(subject_epoch)): 
        # extract epoch  
        epoch_zygo = np.squeeze(subject_epoch[t].get_data(picks=['Zygo']))

        epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo = get_features(epoch_zygo)


        # fill dataframe 
        features.loc[t] = [
            subject, 
            t + 1,
            epoch_wl_zygo,
            epoch_var_zygo, 
            epoch_rms_zygo, 
            epoch_zygo, 
        ]
    
    return features 

## Extracting features 

In [ ]:
i = 0 
features_all = []

# walk through directory to pre-processing and extract features
for root,dirs,files in os.walk(raw_path):
    for file in files:
        if "vhdr" in file and "sleep" in file: # so only take each subject once 
            subject = file.split("_")[0] 
            print(f"Processing{'.'*65}{subject}")

            
            if subject=="Cami":
                filename = subject+"_sleep.vhdr"
            else: 
                filename = subject+'_sleep_active.vhdr'
        
            
            subject_epoch, _ = pre_process_subjets(subject,filename)
            features = make_features_df(subject_epoch,subject)
            features_all.append(features)
        

In [ ]:
features_all_df = pd.concat(features_all, ignore_index=True)


In [ ]:
features_all_df.to_excel("training_features.xlsx", index=False)

## GUI

In [ ]:
subject_plt = "S02"
subject_df = features_all_df.loc[(features_all_df["Subject"] == subject_plt)] # extract subj info

In [ ]:
#GUI
current_index=0
inter_trigger_length=10
window = 50
step = 1

def plot_figure(t):    
    epoch_zygo = np.array(subject_df["Zygo"].tolist())*10000 # scale 

    # can also plot features 
    epoch_zygo_rms = np.array(subject_df["Zygo"].tolist())*10000
    epoch_zygo_var = np.array(subject_df["Var_Zygo"].tolist())*10000
    epoch_zygo_wl = np.array(subject_df["WL_Zygo"].tolist())*10000


    fig, ax1 = plt.subplots(1,1, figsize=(20, 4))

    fig.suptitle(f"Epoch {t + 1}")


    # Zygo subplot
    ax1.plot(epoch_zygo[t], label="Zygo",color="black")
    
    #ax1.plot(epoch_zygo_var[t], label="Variance",color="blue")
    #ax1.plot(epoch_zygo_rms[t], label="RMS",color="red")
    #ax1.plot(epoch_zygo_wl[t], label="WL",color="green")
    ax1.set_ylim(-200, 200)
    ax1.set_ylabel("Zygo EMG [V]")
    ax1.set_xlabel("Samples")
 
    # Connect mouse key press events
    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.tight_layout()
    plt.suptitle(f"Subj. {subject_plt} | Epoch {t}")
    plt.show()


# Keyboard press event handler
def on_key(event):
    global current_index, fig, features_all
    key = event.key
        
    if event.key == 'right':  # Move to next figure
        current_index = (current_index + 1) % len(subject_df)  # Loop to the start
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the next figure
    elif event.key == 'left':  # Move to previous figure
        current_index = (current_index - 1) % len(subject_df)  # Loop to the end
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'escape':  
        print("Quitting the plot!")
        plt.close(fig)  
        
# Plot the first figure
plot_figure(38)
